# 📈 ROC EĞRİSİ & AUC SKORU
> **🎯 AMAÇ:** Sınıflandırma modelinin TPR/FPR dengesi ve AUC ile değerlendirilmesi  
> **📥 GİRDİ:** X_train, X_test, y_train, y_test  
> **📤 ÇIKTI:** ROC eğrisi, AUC skoru, model karşılaştırma  
### ⏱️ NE ZAMAN KULLANILIR?
- Sınıf dengesizliği varsa (accuracy yanıltıcı olduğunda)
- Eşik değeri belirlemek için
### 🔧 TEMEL KAVRAMLAR
- **TPR (Recall):** TP/(TP+FN)  |  **FPR:** FP/(FP+TN)
- **AUC=1.0:** Mükemmel  |  **AUC=0.5:** Rastgele

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, accuracy_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
print('=' * 50 + '\n📈 ROC AUC BAŞLATILIYOR\n' + '=' * 50)

In [ ]:
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Logistic Reg' : LogisticRegression(max_iter=1000, random_state=42),
    'SVM (prob)'   : SVC(probability=True, random_state=42)
}
for name, m in models.items():
    m.fit(X_train, y_train.values.ravel())
    print(f'✅ {name} eğitildi.')

In [ ]:
plt.figure(figsize=(8, 6))
colors = ['#2196F3', '#FF5722', '#4CAF50']
for (name, model), color in zip(models.items(), colors):
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={roc_auc:.3f})')
    print(f'{name:<16}: AUC = {roc_auc:.4f}')
plt.plot([0,1],[0,1],'--', color='gray', label='Rastgele (AUC=0.5)')
plt.xlabel('FPR'); plt.ylabel('TPR (Recall)')
plt.title('ROC Eğrisi', fontsize=14, fontweight='bold')
plt.legend(loc='lower right'); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Optimal eşik - Youden's J
best_model = list(models.values())[0]
best_name  = list(models.keys())[0]
y_prob = best_model.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
optimal_idx = np.argmax(tpr - fpr)
opt_thr = thresholds[optimal_idx]
print(f'\n🔍 {best_name} Optimal Eşik: {opt_thr:.4f}')
print(f'   TPR: {tpr[optimal_idx]:.4f} | FPR: {fpr[optimal_idx]:.4f}')
y_pred_opt = (y_prob >= opt_thr).astype(int)
print(f'   Accuracy: %{accuracy_score(y_test, y_pred_opt)*100:.2f}')
print(f'   F1-Score: {f1_score(y_test, y_pred_opt):.4f}')
print('\n✅ İşlem Tamamlandı.')